# Fourier series and the Gibbs phenomenon

*maths · beginner* · [open in Tensorizer](https://jackyko1991.github.io/tensorizer/#/maths/fourier_gibbs)

A square wave, a sawtooth and a triangle wave built from sines. The coefficients are one einsum (a numerical integral, checked against $4/(\pi n)$ and friends) and every partial sum is one matrix product $S_N = B\,c$ with the first $N$ columns of the basis matrix. Watch the wiggles crowd into the jumps while the overshoot stays at about 9% of the jump.

**Vector form**

$$f(x) \approx S_N(x) = \sum_{n=1}^{N} b_n \sin(n x),\qquad b_n = \frac{1}{\pi}\int_{-\pi}^{\pi} f(x)\,\sin(n x)\,dx$$

**Index (tensor) form**

$$B_{jn} = \sin(n\,x_j),\qquad c_{wn} = \tfrac{1}{\pi}\,w_j\,F_{wj}\,B_{jn},\qquad S_{wj} = B_{jn}\,c_{wn}\ \ (n \le N),\qquad E_w = \tfrac{1}{\pi}\,w_j\,(F_{wj} - S_{wj})^2$$

| Symbol | Code | Shape | Meaning |
|---|---|---|---|
| $x_j,\ w_j$ | `grid(n_x, device)` | (X,), (X,) | midpoints of $X$ equal cells on $(-\pi, \pi)$ and their width $h$: the integration weights |
| $F_{wj}$ | `waves(x)` | (3, X) | wave $w$ (square, sawtooth, triangle) sampled at $x_j$ |
| $B_{jn}$ | `basis(x, n_max)` | (X, N) | basis matrix: column $n$ is $\sin(n x)$ on the grid |
| $c_{wn}$ | `einsum('j,wj,jn->wn', w, F, B) / pi` | (3, N) | repeated $j$ ⇒ summed: the integral $b_n$ as a weighted sum over the grid |
| $S_{wj}$ | `einsum('jn,wn->wj', B[:, :N], c[:, :N])` | (3, X) | repeated $n$ ⇒ summed: the partial sum with $N$ harmonics, a matrix product |
| $E_w$ | `einsum('j,wj,wj->w', w, d, d) / pi` | (3,) | squared $L^2$ error with $d = F - S$; repeated $j$ ⇒ summed |

The exact coefficients are $b_n = 4/(\pi n)$ for odd $n$ (square), $2(-1)^{n+1}/(\pi n)$ (sawtooth) and $\pm 8/(\pi^2 n^2)$ for odd $n$ (triangle); the numerical integrals match them to about $10^{-6}$. Near a jump the partial sums overshoot. As $N$ grows the bump moves towards the jump (its width shrinks like $1/N$) but its height tends to $\left(\tfrac{1}{\pi}\mathrm{Si}(\pi) - \tfrac12\right) \approx 0.0895$ times the jump, not to zero: for the square wave from $-1$ to $1$ the peak stays near $1.179$. This is the Gibbs phenomenon. The series still converges in the mean: by Parseval, $\tfrac1\pi\int f^2\,dx = \sum_n b_n^2$, so the squared $L^2$ error is the sum of the missing $b_n^2$. Coefficients falling like $1/n$ (a jump) give an error $\propto 1/N$; the triangle has only a kink, its coefficients fall like $1/n^2$ and the error like $1/N^3$. Smoother functions converge faster. Try a coarse grid (small $X$) to see the numerical coefficients drift.


## Setup

On Colab, choose **Runtime → Change runtime type → GPU** to run on a GPU — the code uses it automatically.

In [ ]:
import os
import subprocess
import sys

# Shared helpers (problems/_common) come from the repository; skip the clone if we are already inside it.
if not os.path.isdir("problems"):
    if not os.path.isdir("tensorizer"):
        subprocess.run(["git", "clone", "--depth", "1", "https://github.com/jackyko1991/tensorizer.git"], check=True)
    os.chdir("tensorizer")
sys.path.insert(0, os.getcwd())
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "pyyaml", "tqdm"], check=True)

import torch

print("torch", torch.__version__, "· device:", "cuda" if torch.cuda.is_available() else "cpu")


## 1. `main.py` — main loop

In [ ]:
%%writefile problems/maths/fourier_gibbs/main.py
"""Main loop: parameters → tensors → solve → plot."""

from pathlib import Path

from problems._common.runtime import cli, device

from . import plot, solve, tensors
from .equation import coefficients, energy

GIBBS = 0.0895  # (1/π) Si(π) − 1/2: the Wilbraham–Gibbs constant


def run(params: dict, out_dir: Path) -> Path:
    dev = device()
    n_max = params["n_max"]
    x, w = tensors.grid(params["n_x"], dev)  # (X,), (X,)
    f = tensors.waves(x)  # (3, X)
    b = tensors.basis(x, n_max)  # (X, N)
    print(f"{f.shape[0]} waves on {x.shape[0]} grid points, basis matrix B of shape {tuple(b.shape)}")

    c = coefficients(f, b, w)  # (3, N)
    exact = tensors.exact_coefficients(n_max, dev)
    for k, name in enumerate(tensors.WAVES):
        first = ", ".join(f"{v:+.4f}" for v in c[k, :5].tolist())
        print(f"{name:8s} b_1..b_5 = {first}   max |numerical − exact| = {(c[k] - exact[k]).abs().max():.1e}")
    tail = (energy(f, w) - (c**2).sum(dim=1)).tolist()
    print("Parseval (1/π)∫f² − Σ b_n² (what the first N_max harmonics miss): " + ", ".join(f"{t:.1e}" for t in tail))

    s = solve.all_partial_sums(b, c)  # (N, 3, X)
    over = solve.overshoot(s, f, tensors.JUMP)  # (N, 3)
    err = solve.errors(f, s, w)  # (N, 3)
    for n in (1, 9, n_max // 2, n_max):
        n = n if n % 2 == 1 else n - 1  # odd N: the square wave's even harmonics are 0
        print(f"N = {n:4d}: overshoot / jump  square {over[n - 1, 0]:.4f}  sawtooth {over[n - 1, 1]:.4f}")
    print(f"Gibbs: the overshoot tends to {GIBBS} × jump, not to 0 (only its width shrinks, like 1/N)")
    rate = solve.decay_rate(err)
    print("L² error² ∝ N^(−p):  " + "  ".join(f"{nm} p = {p:.2f}" for nm, p in zip(tensors.WAVES, rate.tolist())))
    return plot.animate(x, f, s, over, err, out_dir / "fourier_gibbs.gif")


if __name__ == "__main__":
    cli(run, __file__)


## 2. `equation.py` — physics → tensor translation

In [ ]:
%%writefile problems/maths/fourier_gibbs/equation.py
r"""Maths → tensor translation.

A periodic odd function on (−π, π) is a sum of sines, f(x) = \sum_n b_n \sin(nx), with coefficients

    b_n = \frac{1}{\pi}\int_{-\pi}^{\pi} f(x)\,\sin(nx)\,dx.

On a grid x_j with cell widths w_j the integral becomes a sum. Put the sampled waves in F_{wj} (w = which wave,
j = grid point) and the basis in a matrix B_{jn} = sin(n x_j); then all coefficients of all waves are one einsum,

    c_{wn} = \tfrac{1}{\pi}\,w_j\,F_{wj}\,B_{jn}          (j summed),

and the partial sum with the first N harmonics is one matrix product, S^{(N)}_{wj} = B_{jn} c_{wn} for n ≤ N
(n summed). The squared L² error is E_w = \tfrac{1}{\pi} w_j (F_{wj} − S_{wj})² (j summed).
"""

import math

import torch


def coefficients(f: torch.Tensor, b: torch.Tensor, w: torch.Tensor) -> torch.Tensor:
    r"""c_{wn} = (1/π) w_j F_{wj} B_{jn}   (j summed).   F (W, X), B (X, N), w (X,) → (W, N)"""
    return torch.einsum("j,wj,jn->wn", w, f, b) / math.pi


def partial_sum(b: torch.Tensor, c: torch.Tensor, n: int) -> torch.Tensor:
    r"""S^{(N)}_{wj} = B_{jn} c_{wn} over the first N harmonics   (n summed).   → (W, X)"""
    return torch.einsum("jn,wn->wj", b[:, :n], c[:, :n])


def l2_error_sq(f: torch.Tensor, s: torch.Tensor, w: torch.Tensor) -> torch.Tensor:
    r"""E_w = (1/π) w_j (F_{wj} − S_{wj})²   (j summed).   → (W,)"""
    d = f - s
    return torch.einsum("j,wj,wj->w", w, d, d) / math.pi


def energy(f: torch.Tensor, w: torch.Tensor) -> torch.Tensor:
    r"""(1/π) ∫ f² dx = (1/π) w_j F_{wj} F_{wj}: the left side of Parseval's identity   → (W,)"""
    return torch.einsum("j,wj,wj->w", w, f, f) / math.pi


## 3. `tensors.py` — PyTorch tensors

In [ ]:
%%writefile problems/maths/fourier_gibbs/tensors.py
"""Build the concrete tensors: the sample grid and weights, the three waves, the sine basis, exact coefficients."""

import math

import torch

WAVES = ["square", "sawtooth", "triangle"]
JUMP = [2.0, 2.0, 0.0]  # size of the jump in each wave (the triangle is continuous)


def grid(n_x: int, device: torch.device) -> tuple[torch.Tensor, torch.Tensor]:
    """Midpoints x_j of n_x equal cells on (−π, π) and their widths w_j = h: (X,), (X,). No point sits on a jump."""
    h = 2 * math.pi / n_x
    x = -math.pi + h * (torch.arange(n_x, dtype=torch.float64, device=device) + 0.5)
    return x, torch.full_like(x, h)


def waves(x: torch.Tensor) -> torch.Tensor:
    """F_{wj}: the square, sawtooth and triangle waves at x_j. All are odd and swing between −1 and 1: (3, X)."""
    square = torch.sign(torch.sin(x))
    sawtooth = x / math.pi
    triangle = 2 / math.pi * torch.asin(torch.sin(x))
    return torch.stack([square, sawtooth, triangle])


def basis(x: torch.Tensor, n_max: int) -> torch.Tensor:
    """B_{jn} = sin(n x_j) for the harmonics n = 1 … N_max: (X, N_max). Odd waves need only sines."""
    n = torch.arange(1, n_max + 1, dtype=x.dtype, device=x.device)  # (N_max,)
    return torch.sin(x[:, None] * n[None, :])


def exact_coefficients(n_max: int, device: torch.device) -> torch.Tensor:
    """The textbook b_n of the three waves, to check the numerical integrals: (3, N_max)."""
    n = torch.arange(1, n_max + 1, dtype=torch.float64, device=device)
    odd = (n % 2 == 1).double()
    square = 4 / (math.pi * n) * odd  # 4/(πn) for odd n, 0 for even n
    sawtooth = 2 / (math.pi * n) * torch.where(n % 2 == 1, 1.0, -1.0)  # 2(−1)^{n+1}/(πn)
    triangle = 8 / (math.pi * n) ** 2 * torch.where(n % 4 == 1, 1.0, -1.0) * odd  # ±8/(π²n²) for odd n
    return torch.stack([square, sawtooth, triangle])


## 4. `solve.py` — solver

In [ ]:
%%writefile problems/maths/fourier_gibbs/solve.py
"""Build every partial sum S_1 … S_Nmax and measure the Gibbs overshoot and the L² error of each."""

import torch

from problems._common.progress import steps

from .equation import l2_error_sq, partial_sum


def all_partial_sums(b: torch.Tensor, c: torch.Tensor) -> torch.Tensor:
    """S[N-1] = partial sum with N harmonics, for N = 1 … N_max: (N_max, W, X)."""
    return torch.stack([partial_sum(b, c, n + 1) for n in steps(c.shape[1], "partial sums")])


def overshoot(s: torch.Tensor, f: torch.Tensor, jump: list[float]) -> torch.Tensor:
    """At the highest point of S_N: how far it rises above the wave there, as a fraction of the jump.

    s (N_max, W, X), f (W, X) → (N_max, W). Raw S − f for a wave without a jump.
    """
    peak = s.argmax(dim=-1)  # (N_max, W): grid index of the top of each partial sum
    above = s.amax(dim=-1) - f[torch.arange(f.shape[0]), peak]
    scale = torch.tensor([j if j > 0 else 1.0 for j in jump], dtype=s.dtype, device=s.device)
    return above / scale


def errors(f: torch.Tensor, s: torch.Tensor, w: torch.Tensor) -> torch.Tensor:
    """Squared L² error of every partial sum: (N_max, W)."""
    return torch.stack([l2_error_sq(f, s_n, w) for s_n in s])


def decay_rate(err: torch.Tensor) -> torch.Tensor:
    """Fitted p in E_N ∝ N^(−p), using odd N in the upper half of the range (even harmonics of the square are 0).

    err (N_max, W) → (W,)
    """
    n = torch.arange(1, err.shape[0] + 1, dtype=err.dtype, device=err.device)
    keep = (n % 2 == 1) & (n > err.shape[0] // 2)
    x = torch.log(n[keep])  # (K,)
    y = torch.log(err[keep])  # (K, W)
    x = x - x.mean()
    slope = torch.einsum("k,kw->w", x, y - y.mean(dim=0)) / (x @ x)
    return -slope


## 5. `plot.py` — 2D/3D plot → GIF

In [ ]:
%%writefile problems/maths/fourier_gibbs/plot.py
"""Animate the partial sums of the three waves as N grows, with the Gibbs overshoot and the L² error below."""

from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import torch

from problems._common.plotting import frame_indices, save_gif

from .tensors import WAVES

COLORS = ["#2563eb", "#dc2626", "#16a34a"]
GIBBS = 0.0895


def animate(
    x: torch.Tensor, f: torch.Tensor, s: torch.Tensor, over: torch.Tensor, err: torch.Tensor, path: Path
) -> Path:
    """x (X,), f (3, X), s (N, 3, X), over (N, 3), err (N, 3)."""
    x, f, s = x[::2].cpu().numpy(), f[:, ::2].cpu().numpy(), s[:, :, ::2].cpu().numpy()  # every 2nd point is enough
    over, err = over.cpu().numpy(), err.cpu().numpy()
    n_max = s.shape[0]
    ns = np.arange(1, n_max + 1)
    odd = ns % 2 == 1  # plot odd N only: the square wave's even harmonics are 0, so its curves would zigzag

    fig = plt.figure(figsize=(9, 5.6))
    grid = fig.add_gridspec(2, 6, height_ratios=[1, 1.1])
    tops = [fig.add_subplot(grid[0, 2 * k : 2 * k + 2]) for k in range(3)]
    ax_g, ax_e = fig.add_subplot(grid[1, :3]), fig.add_subplot(grid[1, 3:])

    curves = []
    for k, ax in enumerate(tops):
        ax.plot(x, f[k], color="#9ca3af", lw=1.2)
        (line,) = ax.plot(x, s[0, k], color=COLORS[k], lw=1.2)
        curves.append(line)
        if k < 2:  # the top of the overshoot that never goes away: 1 + 0.0895 × jump
            ax.axhline(1 + 2 * GIBBS, color="#f59e0b", lw=1, ls="--")
        ax.set_xlim(-np.pi, np.pi)
        ax.set_ylim(-1.4, 1.4)
        ax.set_xticks([-np.pi, 0, np.pi], ["−π", "0", "π"])
        ax.set_title(WAVES[k], fontsize=10)
    title = fig.suptitle("", fontsize=11)

    trails_g, trails_e = [], []
    for k in range(3):
        if k < 2:
            (tg,) = ax_g.plot([], [], color=COLORS[k], lw=1.4, label=WAVES[k])
            trails_g.append(tg)
        (te,) = ax_e.plot([], [], color=COLORS[k], lw=1.4, label=WAVES[k])
        trails_e.append(te)
    ax_g.axhline(GIBBS, color="#f59e0b", lw=1.2, ls="--", label="0.0895 (Gibbs)")
    ax_g.axhline(0, color="#374151", lw=0.8)
    ax_g.set_xlim(1, n_max)
    ax_g.set_ylim(-0.01, 0.16)
    ax_g.set_xlabel("harmonics N")
    ax_g.set_ylabel("overshoot / jump")
    ax_g.set_title("overshoot does not go to 0", fontsize=10)
    ax_g.legend(fontsize=8, loc="upper right")

    ref = ns[odd].astype(float)  # reference trends, drawn a little above the last odd-N errors
    last = ref[-1]
    ax_e.loglog(ref, 2 * err[odd][-1, 0] * last / ref, color="#9ca3af", ls="--", lw=1, label="∝ 1/N")
    ax_e.loglog(ref, 2 * err[odd][-1, 2] * (last / ref) ** 3, color="#9ca3af", ls=":", lw=1.2, label="∝ 1/N³")
    ax_e.set_xlim(1, n_max)
    ax_e.set_ylim(max(err[odd].min() / 3, 1e-14), err.max() * 3)
    ax_e.set_xlabel("harmonics N")
    ax_e.set_ylabel("‖f − S_N‖²")
    ax_e.set_title("squared L² error", fontsize=10)
    ax_e.legend(fontsize=8, loc="lower left", ncol=2)

    def update(i: int):
        n = i + 1
        for k in range(3):
            curves[k].set_ydata(s[i, k])
        shown = odd & (ns <= n)
        for k in range(2):
            trails_g[k].set_data(ns[shown], over[shown, k])
        for k in range(3):
            trails_e[k].set_data(ns[shown], err[shown, k])
        title.set_text(f"Fourier partial sums  $S_N = B_{{[:, :N]}}\\,c_{{[:N]}}$,   N = {n}")

    fig.tight_layout()
    frames = frame_indices(n_max, 60)
    return save_gif(fig, update, frames + frames[-1:] * 6, path, fps=8)


## Run

Change the parameters (or any file above) and run again.

In [ ]:
import importlib
from pathlib import Path

from IPython.display import Image, display

from problems._common.runtime import load_meta, resolve_params

params = {
    'n_max': 100,
    'n_x': 4096
}

# Re-import so edits in the %%writefile cells above take effect.
for name in [m for m in sys.modules if m.startswith("problems.maths.fourier_gibbs")]:
    del sys.modules[name]
main = importlib.import_module("problems.maths.fourier_gibbs.main")
problem_dir = Path("problems/maths/fourier_gibbs")
output = main.run(resolve_params(load_meta(problem_dir), params), Path("out"))
display(Image(filename=str(output)))
